In [15]:
import numpy as np

A = np.array([[1, 2],
              [3, 4]])

B = np.array([[5, 6],
              [7, 8]])

C = np.einsum('ij,kj->ik', A, B)

print(C)
A = np.array([[1, 2],
              [3, 4]])

B = np.array([[5, 6],
              [7, 8]])

C = np.einsum('ij,jk->ik', A, B)

print(C)

A = np.random.rand(2, 3, 4)
B = np.random.rand(4, 5)

C = np.einsum('abc,cd->abd', A, B)
print(C)
print(C.shape)  # (2, 3, 5)

D = 2  # bond dimension
p = 1  # physical dim (scalar output)

T1 = np.random.rand(p, D)
T2 = np.random.rand(D, D)
T3 = np.random.rand(D, p)

# contract everything
result = np.einsum('ai,ij,jb->ab', T1, T2, T3)

print(result)

N = 20   # length
D = 3    # bond dimension

# random tensors
tensors = [np.random.randn(D, D) for _ in range(N)]
tensors[0] = np.random.randn(1, D)
tensors[-1] = np.random.randn(D, 1)

# contract sequentially
state = tensors[0]
for t in tensors[1:]:
    state = np.einsum('ai,ij->aj', state, t)

print(state.shape)

[[17 23]
 [39 53]]
[[19 22]
 [43 50]]
[[[0.33138708 0.39250561 0.27652542 0.34722092 0.17308665]
  [1.6096295  2.23729188 1.37671492 1.39258707 1.43394857]
  [1.60102285 1.9134684  1.36645038 0.73996932 1.1369755 ]]

 [[0.80662812 1.16251477 0.63280627 1.16877635 0.6170317 ]
  [1.26348923 1.58950366 1.17464007 0.50046204 1.09586547]
  [1.84895745 2.59635277 1.57866975 1.60447639 1.68352502]]]
(2, 3, 5)
[[0.24833922]]
(1, 1)


In [17]:
D = 2

# four tensors
T00 = np.random.rand(D, D)  # simplified
T01 = np.random.rand(D, D)
T10 = np.random.rand(D, D)
T11 = np.random.rand(D, D)

top = np.einsum('ab,bc->ac', T00, T01)
bottom = np.einsum('ab,bc->ac', T10, T11)

result = np.einsum('ab,ab->', top, bottom)

print(result)

0.7063702292080224


In [15]:
import numpy as np

N = 32
D = 2
chi = 8
p = 1

rng = np.random.default_rng(42)
def InitPEPS(N, D, p):
    tensors = rng.standard_normal((N, N, D, D, D, D, p)) * 0.1
    return tensors

def ContractRow(row_tensors, boundary_up, boundary_down):
    N_cols = row_tensors.shape[0]
    values = np.zeros(N_cols)
    
    # Left boundary: start with a D-dim vector of ones (open BC)
    left_vec = np.zeros(D)
    left_vec[0] = 1.0
    
    for col in range(N_cols):
        T = row_tensors[col]           # shape (D, D, D, D, p)
        up   = boundary_up[col]        # shape (D,)
        down = boundary_down[col]      # shape (D,)
        
        # Contract: sum over left, up, down indices
        contracted = np.einsum('lrudp,l,u,d->rp', T, left_vec, up, down)
        
        # Read out the scalar value (p=1)
        values[col] = contracted[0, 0]   # right bond index 0, physical 0
        
        # Propagate the right bond as the new left bond
        left_vec = contracted[:, 0]
    
    return values
tensors = InitPEPS(N, D, p)
print(tensors.shape)

boundary_up = np.ones((tensors.shape[0], D))
boundary_down = np.ones((tensors.shape[0], D))
# normalize boundaries
boundary_up /= np.linalg.norm(boundary_up, axis=1, keepdims=True)
boundary_down /= np.linalg.norm(boundary_down, axis=1, keepdims=True)

def ContractPEPSSimple(tensors, D, chi):
    """
    Simple row-by-row contraction with boundary truncation.
    tensors: shape (N, N, D, D, D, D, p)
    Returns: noise_map of shape (N, N)
    """
    N = tensors.shape[0]
    noise_map = np.zeros((N, N))
    
    # Initialize top boundary: uniform vectors
    # boundary[col] has shape (chi,) representing the accumulated state
    boundary = np.ones((N, D)) / D
    
    for row in range(N):
        row_t = tensors[row]  # shape (N, D, D, D, D, p)
        
        # Use current boundary as "up" vectors
        boundary_up   = boundary          # shape (N, D)
        boundary_down = np.ones((N, D)) / D  # flat bottom boundary
        
        # Contract this row
        row_values = ContractRow(row_t, boundary_up, boundary_down)
        noise_map[row] = row_values
        
        # Update boundary by absorbing this row (simplified)
        new_boundary = np.zeros((N, D))
        for col in range(N):
            T = row_t[col]
            up = boundary_up[col]
            # Absorb: sum over up and left, keep down as new boundary
            new_boundary[col] = np.einsum('lrudp,u->d', T, up).reshape(D)
            # Normalize to prevent blow-up
            norm = np.linalg.norm(new_boundary[col])
            if norm > 1e-10:
                new_boundary[col] /= norm
        
        boundary = new_boundary
    
    return noise_map

noise = ContractPEPSSimple(tensors, D, chi)
print(noise.shape)

(32, 32, 2, 2, 2, 2, 1)
(32, 32)
